# 고령 부모 돌봄 대화 어시스턴트 — 로컬 AI 파이프라인

**Main Quest 2** · 음성인식 · 음성생성 · 객체탐지 · 세그멘테이션

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nike1137-svg/poc-elder-care-local-ai/blob/main/notebooks/poc_demo.ipynb)

---

## ▶ 실행하는 법 (처음 오셨다면 여기부터)

**GitHub에서 보고 계신다면 위의 파란 `Open in Colab` 배지를 누르세요.**
구글 계정만 있으면 설치 없이 브라우저에서 바로 돌아갑니다.

Colab이 열리면 순서는 이렇습니다.

| | 할 일 | 걸리는 시간 |
|---|---|---|
| 1 | **런타임 → 런타임 유형 변경 → T4 GPU → 저장** (선택, 권장) | 30초 |
| 2 | **런타임 → 모두 실행** (또는 셀마다 `Shift+Enter`) | — |
| 3 | 첫 두 셀에서 저장소를 받고 패키지를 설치합니다 | **3-5분** |
| 4 | 나머지 셀이 순서대로 돕니다 | 5분 내외 |

> **GPU를 안 켜도 전부 동작합니다.** 이 프로젝트의 요점이 "GPU 없이 돌아간다"라서,
> CPU에서도 끝까지 실행됩니다. GPU를 켜면 더 큰 모델(`large-v3`)로 자동 전환됩니다.

> **중간에 멈춘 것처럼 보이는 구간이 있습니다.** 설치 셀에서 패키지를 푸는 동안
> 몇 분간 출력이 없습니다. 정상입니다.

> **§8(기존 방식 비교)은 건너뛰어도 됩니다.** API 키를 물어보면 그냥 엔터를 치세요.
> 키가 없으면 그 절만 건너뛰고 나머지는 그대로 진행됩니다.

**로컬에서 열었다면** 저장소 안에서 Jupyter로 열면 그대로 돌아갑니다.
설치는 `pip install -r requirements.txt` 로 대신하세요.

---

## 이 노트북이 보여주는 것

```
음성(m4a) → 로컬 Whisper 전사 → 특이사항 추출 → 보호자용 음성 리포트
사진      → YOLO 인물 탐지 → SAM 정밀 분할 → 모자이크
```

| 절 | 내용 |
|---|---|
| §3-4 | 음성 한 건 전사 → 특이사항 추출 |
| §5-6 | 스타일별 성능, 실제로 틀린 사례 |
| §7 | 특이사항 추출 채점 |
| §8 | (선택) 기존 클라우드 방식과 비교 |
| §9-10 | 하이퍼파라미터 튜닝과 **교차 검증** |
| §11 | 강건성 — 일부러 무너뜨려 보기 |
| §12-13 | 사진 마스킹 · 로컬 음성 합성 |

---

## ★ 데이터 취급 원칙

**이 노트북은 실제 사용자의 음성을 담고 있지 않습니다.**

실서비스는 고령의 가족과 나누는 실제 대화를 다루므로, 그 음성은 공개 저장소에
올라갈 수 없습니다. 평가에 쓰는 음성은 전량 **창작 대본을 TTS로 합성한 것**이고,
등장하는 인물·증상은 모두 가공입니다.

→ 자세한 논의는 `docs/problem-statement.md` §0

---

## 무엇을 개선하는가

| | 기존 방식 | 개선안 |
|---|---|---|
| 전사 주체 | 클라우드 범용 LLM에 프롬프트로 요청 | 로컬 Whisper (ASR 전용) |
| 음성 외부 전송 | **있음** | **없음** |
| 비용 | 호출당 과금 | **0원** |
| 실패 처리 | 빈 문자열로 조용히 삼킴 | 원인별로 분류해 보고 |
| 전사 이후 | 없음 | **특이사항 추출** |


## 1. 환경 준비

Colab에서는 아래 셀이 저장소를 내려받고 의존성을 설치한다.
로컬에서 이미 저장소 안에 있다면 clone 단계는 자동으로 건너뛴다.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/nike1137-svg/poc-elder-care-local-ai.git"

IN_COLAB = "google.colab" in sys.modules
print("Colab 환경" if IN_COLAB else "로컬 환경")

# 이미 저장소 안에서 열었다면 그 위치를 쓴다 (로컬 실행)
def find_root(start: pathlib.Path):
    for p in [start, *start.parents]:
        if (p / "src" / "pipeline.py").exists():
            return p
    return None

ROOT = find_root(pathlib.Path.cwd())

# 없으면 내려받는다 (Colab에서 이 경로를 탄다)
if ROOT is None:
    target = pathlib.Path("poc-elder-care-local-ai")
    if not target.exists():
        print("저장소를 내려받는 중...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    ROOT = target.resolve()

print("저장소:", ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)


### 패키지 설치

**약 3-5분 걸립니다.** 중간에 `Installing collected packages: ...` 가 뜬 뒤
몇 분간 출력이 멈춘 것처럼 보이는데 **정상**입니다. 패키지를 푸는 중입니다.


In [ ]:
# 노트북이 쓰는 패키지 전부
#   음성:   faster-whisper(전사) · piper-tts(합성) · edge-tts(평가용 음성 생성)
#   이미지: ultralytics(YOLO/SAM) · opencv-python
#   평가:   jiwer(CER)
# ※ 회상 이미지(diffusers)는 이 노트북에서 다루지 않아 제외했습니다.
%pip install -q faster-whisper piper-tts edge-tts jiwer soundfile ultralytics opencv-python

# ffmpeg 확인 (Colab에는 기본 설치되어 있다)
import shutil
if shutil.which("ffmpeg") is None:
    !apt-get -qq install -y ffmpeg
print("ffmpeg:", shutil.which("ffmpeg"))


In [ ]:
# GPU가 있으면 자동으로 쓴다. 없으면 CPU int8로 돈다.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

DEVICE = "cuda" if HAS_GPU else "cpu"
COMPUTE = "float16" if HAS_GPU else "int8"
MODEL_SIZE = "large-v3" if HAS_GPU else "small"

print(f"장치: {DEVICE} / 연산타입: {COMPUTE} / 모델: {MODEL_SIZE}")
if not HAS_GPU:
    print("\n※ GPU가 없어 small 모델로 돕니다.")
    print("   Colab에서 GPU를 켜려면: 런타임 → 런타임 유형 변경 → T4 GPU")

## 2. 평가용 음성 준비

저장소에 합성 음성이 이미 들어 있다. 없으면 대본에서 다시 만든다.
**실사용자 음성은 어디에도 없다.**

In [ ]:
import json, pathlib

corpus = json.loads((ROOT / "data" / "reference" / "corpus.json").read_text(encoding="utf-8"))
audio_dir = ROOT / "data" / "audio"
existing = sorted(audio_dir.glob("*.m4a")) if audio_dir.exists() else []

print(f"대본 {len(corpus['utterances'])}건 / 음성 파일 {len(existing)}건")

if len(existing) < len(corpus["utterances"]):
    print("음성이 부족합니다. 대본에서 합성합니다...")
    !python src/build_dataset.py
else:
    print("음성이 모두 준비되어 있습니다.")

print()
print("대본 예시:")
for u in corpus["utterances"][:4]:
    kinds = ", ".join(s["type"] for s in u["signals"])
    print(f"  {u['id']} [{u['style']:6}] {u['text']}")
    print(f"        심어둔 신호: {kinds}")

## 3. 파이프라인 한 건 돌려보기

음성 하나를 넣어 전사와 특이사항 리포트가 나오는 것을 확인한다.

In [ ]:
from stt_whisper import WhisperBackend
from extract_signals import extract, format_report
import time

backend = WhisperBackend(model_size=MODEL_SIZE, compute_type=COMPUTE, device=DEVICE)

print(f"모델 적재 중... ({MODEL_SIZE})  최초 실행은 내려받느라 시간이 걸립니다.")
t0 = time.perf_counter()
backend.warmup()
print(f"적재 완료 {time.perf_counter() - t0:.1f}s")

In [ ]:
# 들어보기 (Colab에서 재생 가능)
from IPython.display import Audio, display

SAMPLE = "u13"   # 속이 안 좋아 아침을 거른 상황
path = ROOT / "data" / "audio" / f"{SAMPLE}.m4a"
display(Audio(str(path)))

gold = next(u for u in corpus["utterances"] if u["id"] == SAMPLE)
print("정답 대본:", gold["text"])

In [ ]:
res = backend.transcribe(path)

print(f"성공 여부: {res.ok}   (실패면 원인: {res.failure.value})")
print(f"음성 {res.audio_duration_sec:.2f}s → 전사 {res.latency_sec:.2f}s "
      f"(실시간 대비 {res.realtime_factor:.2f}배)")
print()
print("전사 결과:", res.text)
print()
print(format_report(res.text, extract(res.text)))

## 4. 전체 데이터셋 평가 (개선안)

20건 전체를 돌려 CER·지연시간·비용·외부 전송을 측정한다.

In [ ]:
from normalize import cer
import statistics

rows = []
for u in corpus["utterances"]:
    p = ROOT / "data" / "audio" / f"{u['id']}.m4a"
    r = backend.transcribe(p, audio_id=u["id"])
    c = cer(u["text"], r.text) if r.ok else -1
    rows.append({
        "id": u["id"], "style": u["style"], "ok": r.ok,
        "cer": c, "latency": r.latency_sec, "rtf": r.realtime_factor,
        "ref": u["text"], "hyp": r.text,
    })
    mark = "OK" if r.ok else "실패"
    print(f"{u['id']} [{mark}] CER {c:.4f}  {r.latency_sec:.2f}s  {r.text[:44]}")

ok_rows = [r for r in rows if r["ok"]]
cers = [r["cer"] for r in ok_rows]
print()
print("─" * 60)
print(f"성공        {len(ok_rows)}/{len(rows)}")
print(f"CER 평균     {statistics.fmean(cers):.4f}")
print(f"CER 중앙값   {statistics.median(cers):.4f}")
print(f"지연 평균    {statistics.fmean([r['latency'] for r in ok_rows]):.2f}s")
print(f"RTF 평균     {statistics.fmean([r['rtf'] for r in ok_rows]):.3f}")
print(f"외부 전송    0건")
print(f"비용        0원")

## 5. 스타일별 성능

어르신 발화의 특성(느린 말, 문장 중간 침묵, 생활 소음)에서 무너지지 않는지 본다.

In [ ]:
from collections import defaultdict

by_style = defaultdict(list)
for r in ok_rows:
    by_style[r["style"]].append(r["cer"])

print(f"{'스타일':10} {'건수':>4} {'평균 CER':>10}")
print("─" * 28)
for style, vals in sorted(by_style.items()):
    print(f"{style:10} {len(vals):>4} {statistics.fmean(vals):>10.4f}")

## 6. 실패 사례 살펴보기

잘 된 것만 보면 검증이 아니다. 가장 많이 틀린 것부터 본다.

In [ ]:
for r in sorted(ok_rows, key=lambda x: -x["cer"])[:5]:
    if r["cer"] <= 0:
        break
    print(f"[{r['id']}] CER {r['cer']:.4f}  ({r['style']})")
    print(f"  정답: {r['ref']}")
    print(f"  인식: {r['hyp']}")
    print()

## 7. 특이사항 추출 채점 (S5)

대본에 심어둔 신호를 실제로 잡아내는지 센다.

In [ ]:
tp = fn = fp = 0
for u, r in zip(corpus["utterances"], rows):
    goldset = {s["type"] for s in u["signals"]}
    predset = {s.signal_type for s in extract(r["hyp"])} if r["ok"] else set()
    tp += len(goldset & predset)
    fn += len(goldset - predset)
    fp += len(predset - goldset)

recall = tp / (tp + fn) if (tp + fn) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
print(f"TP {tp}  FN {fn}  FP {fp}")
print(f"재현율 {recall:.3f}   정밀도 {precision:.3f}")
print(f"S5 기준(≥0.70): {'달성' if recall >= 0.70 else '미달'}")
print()
print("※ 이 수치는 과대평가다. 대본과 추출 규칙을 같은 사람이 썼기 때문이다.")
print("   자세한 논의는 docs/limitations.md 참조.")

## 8. (선택) 기존 방식과 비교

실서비스가 쓰는 **클라우드 LLM 프롬프트 전사**를 같은 데이터에 돌려 나란히 비교한다.

이 셀은 무료 API 키가 있어야 돈다. 키가 없으면 건너뛰어도 된다
(개선안의 수치는 위에서 이미 다 나왔다).

> 평가 데이터가 **합성 대본**이라 무료 티어를 써도 프라이버시 문제가 없다.
> 실서비스가 무료 티어를 못 쓰는 이유가 바로 "실제 가족 대화라서"였다는 점이
> 이 PoC의 출발점이다.

In [ ]:
import getpass

if not os.environ.get("GEMINI_API_KEY"):
    key = getpass.getpass("GEMINI_API_KEY (없으면 그냥 엔터): ").strip()
    if key:
        os.environ["GEMINI_API_KEY"] = key

HAS_KEY = bool(os.environ.get("GEMINI_API_KEY"))
print("키 있음 — 기준선을 측정합니다." if HAS_KEY else "키 없음 — 이 절은 건너뜁니다.")

In [ ]:
if HAS_KEY:
    from stt_gemini import GeminiBackend

    gb = GeminiBackend()
    base_rows = []
    for u in corpus["utterances"]:
        p = ROOT / "data" / "audio" / f"{u['id']}.m4a"
        r = gb.transcribe(p, audio_id=u["id"])
        c = cer(u["text"], r.text) if r.ok else -1
        base_rows.append({"id": u["id"], "ok": r.ok, "cer": c,
                          "latency": r.latency_sec, "hyp": r.text,
                          "failure": r.failure.value})
        mark = "OK" if r.ok else f"실패({r.failure.value})"
        print(f"{u['id']} [{mark}] CER {c:.4f}  {r.latency_sec:.2f}s  {r.text[:40]}")
else:
    base_rows = []

In [ ]:
if base_rows:
    b_ok = [r for r in base_rows if r["ok"]]
    b_cer = statistics.fmean([r["cer"] for r in b_ok])
    w_cer = statistics.fmean(cers)

    print(f"{'항목':22} {'기존(클라우드 LLM)':>20} {'개선안(로컬 Whisper)':>22}")
    print("─" * 68)
    print(f"{'CER 평균':22} {b_cer:>20.4f} {w_cer:>22.4f}")
    print(f"{'성공 건수':22} {len(b_ok):>20} {len(ok_rows):>22}")
    print(f"{'지연 평균(초)':22} {statistics.fmean([r['latency'] for r in b_ok]):>20.2f} "
          f"{statistics.fmean([r['latency'] for r in ok_rows]):>22.2f}")
    print(f"{'외부 전송(건)':22} {len(base_rows):>20} {0:>22}")
    print(f"{'비용':22} {'호출당 과금':>20} {'0원':>22}")
    print()
    delta = w_cer - b_cer
    print(f"CER 차이: {delta:+.4f}  "
          f"({'개선안이 더 정확' if delta < 0 else '기존이 더 정확'})")
else:
    print("기준선을 측정하지 않았습니다. docs/evaluation.md 에 기록된 측정 결과를 참고하세요.")

## 9. 하이퍼파라미터 튜닝 — 학습 없이 성능 올리기

파인튜닝에는 도메인 음성이 필요한데 그게 바로 안 쓰기로 한 데이터다.
대신 `initial_prompt`로 도메인 어휘를 흘려 **학습 없이** 적응시킨다.

> ⚠️ 같은 데이터로 튜닝하고 같은 데이터로 보고하면 과적합이다.
> 아래 §10에서 공개 데이터셋으로 교차 검증한다.

In [ ]:
from stt_whisper import DOMAIN_PROMPT, WhisperBackend
import statistics

print('도메인 어휘 프롬프트:')
print(' ', DOMAIN_PROMPT)

def measure(prompt, label):
    be = WhisperBackend(model_size=MODEL_SIZE, compute_type=COMPUTE,
                        device=DEVICE, initial_prompt=prompt)
    be.warmup()
    cs = []
    for u in corpus['utterances']:
        r = be.transcribe(ROOT / 'data' / 'audio' / f"{u['id']}.m4a")
        if r.ok:
            cs.append(cer(u['text'], r.text))
    m = statistics.fmean(cs)
    print(f'{label:24} CER {m:.4f}')
    return m

base = measure(None, '튜닝 전 (프롬프트 없음)')
tuned = measure(DOMAIN_PROMPT, '튜닝 후 (도메인 어휘)')
print(f'\n개선율 {(base-tuned)/base*100:.1f}%')

## 10. 교차 검증 — 개선이 진짜인지 확인

도메인 어휘는 돌봄 대화용이다. **뉴스 낭독체인 Zeroth에서는 도움이 적어야 정상**이고,
무엇보다 **악화되지 않아야** 채택할 수 있다.

이 검증이 없었다면 실서비스를 20% 악화시킬 설정을 골랐을 것이다.

In [ ]:
import json as _json, pathlib as _p

f = ROOT / 'results' / 'hyperparam_sweep_zeroth.json'
if f.exists():
    d = _json.loads(f.read_text(encoding='utf-8'))
    print(f"{'설정':22} {'Zeroth CER':>12} {'baseline 대비':>14}")
    print('-'*50)
    b = d['baseline_cer']
    for r in sorted(d['results'], key=lambda x: x['cer_mean']):
        delta = (r['cer_mean']-b)/b*100
        mark = '악화' if delta > 0 else '개선'
        print(f"{r['config']:22} {r['cer_mean']:>12.4f} {delta:>+12.1f}% {mark}")
    print('\n합성 데이터 1등이던 prompt_full_beam1 이 여기서는 최하위다.')
else:
    print('먼저 실행: python src/tune_whisper.py --dataset zeroth')

## 11. 강건성 — 일부러 무너뜨려 보기

무음·소음·손상 파일을 넣었을 때 **없는 말을 지어내는지**가 핵심이다.
환각이 나오면 존재하지 않는 건강 신호가 보호자에게 보고된다.

In [ ]:
import subprocess

subprocess.run([sys.executable, 'src/robustness_test.py'], cwd=str(ROOT))

## 12. 이미지 파이프라인 — 사진도 집 안에서

사진은 음성보다 더 직접적인 개인정보다. 같은 원칙을 적용한다.

- **YOLO → SAM**: 인물을 찾아 모자이크 (프라이버시)
- **SAM 자동분할**: 알약 세기 (복약 확인)

> YOLO를 알약에 못 쓰는 이유: COCO 80개 클래스에 '알약'이 없다.

In [ ]:
from image_pipeline import mask_people, count_pills
from IPython.display import Image, display

r = mask_people(ROOT / 'data' / 'images' / 'person_01.png')
print(f"인물 {len(r['detections'])}명 탐지 → {r['masked_count']}개 마스킹")
print(f"YOLO {r['detect_sec']}s + SAM {r['segment_sec']}s, 외부전송 {r['external_requests']}건")
display(Image(str(ROOT / 'data' / 'images' / 'person_01.png'), width=340))
display(Image(r['output'], width=340))

In [ ]:
import json as _json

gt = _json.loads((ROOT/'data'/'pill_bench'/'ground_truth.json').read_text(encoding='utf-8'))
print(f"{'이미지':12} {'정답':>5} {'예측':>5} {'오차':>6}")
print('-'*32)
errs = []
for e in gt['images']:
    r = count_pills(ROOT/'data'/'pill_bench'/e['file'])
    err = r['pill_count'] - e['ground_truth_count']
    errs.append(abs(err))
    print(f"{e['id']:12} {e['ground_truth_count']:>5} {r['pill_count']:>5} {err:>+6}")
print(f'\n평균 절대 오차 {sum(errs)/len(errs):.2f}개')

## 13. 로컬 TTS — 음성 루프 닫기

기존 TTS(edge-tts)는 로컬 프로세스지만 **합성은 외부에서** 일어난다.
piper로 바꾸면 루프 전체가 집 안에서 닫힌다.

In [ ]:
from pathlib import Path as _Path
import wave, subprocess
from IPython.display import Audio, display

vm = ROOT / 'models' / 'piper' / 'ko_KR-kss-medium.onnx'

# 음성 모델(61MB)은 저장소에 없으므로 없으면 내려받는다
if not vm.exists():
    vm.parent.mkdir(parents=True, exist_ok=True)
    base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/ko/ko_KR/kss/medium/ko_KR-kss-medium.onnx'
    print('음성 모델 내려받는 중... (61MB)')
    subprocess.run(['curl','-sL','-o',str(vm), base], check=True)
    subprocess.run(['curl','-sL','-o',str(vm)+'.json', base+'.json'], check=True)
    print('완료')

from piper import PiperVoice
from voice_report import build_script
from extract_signals import extract as _extract

voice = PiperVoice.load(str(vm))
r = backend.transcribe(ROOT / 'data' / 'audio' / 'u13.m4a')
script = build_script(_extract(r.text), r.text)
print('낭독문:', script)

out = ROOT / 'results' / 'voice' / 'nb_report.wav'
out.parent.mkdir(parents=True, exist_ok=True)
with wave.open(str(out), 'wb') as w:
    voice.synthesize_wav(script, w)
display(Audio(str(out)))


---

## 정리

| 성공 기준 | 목표 | 확인 위치 |
|---|---|---|
| S1 CER | 기존 대비 동등 이상 | §4, §8 |
| S2 처리 시간 | 실시간 1배속 이내 | §4 (RTF) |
| S3 비용 | 0원 | §4 |
| S4 외부 전송 | 0건 | §4 |
| S5 특이사항 재현율 | 70% 이상 | §7 |

측정 결과 해석과 한계는 다음 문서에 있다.

- `docs/evaluation.md` — 검증 결과 전문
- `docs/limitations.md` — 한계와 다음 스텝
- `docs/model-selection.md` — 모델 선정 근거